# demografie-fetch im Jupyter-Notebook

Dieses Notebook zeigt Installation und Nutzung der Komponente: eine Tabelle von Destatis abrufen, das Ergebnis ansehen, den Cache nutzen und mehrere Quellen über eine Konfiguration abrufen.

Voraussetzung: ein Kernel mit **Python 3.11 oder neuer**.

## 1. Installation

`%pip` installiert in die Umgebung des laufenden Kernels. Der Tag legt die Version fest. Ein Neustart des Kernels ist danach nicht nötig.

In [ ]:
%pip install -q "git+https://github.com/trosinde/demografie-fetch.git@v0.2.0"

In [ ]:
import logging

import demografie_fetch
from demografie_fetch import fetch, list_source, run

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")  # Meldungen anzeigen
demografie_fetch.__version__

## 2. Struktur einer Tabelle ansehen

`list_source` lädt keine Daten und schreibt nichts.

In [ ]:
URL = "https://genesis.destatis.de/datenbank/online/statistic/12411/table/12411-0015"
print(list_source(URL))

## 3. Tabelle abrufen

Daten, Cache und Rohfassungen landen unter `data_dir` (Standard: der Ordner des Notebooks). Das Ergebnis zeigt eine Zusammenfassung.

In [ ]:
DATEN = "nb-daten"

r = fetch(URL, startyear=2024, data_dir=DATEN)
r

In [ ]:
r.wide_df.head()  # je Kreis und Stichtag eine Zeile

In [ ]:
r.long_df.head()  # Langformat, eine Zeile je Wert

In [ ]:
r.meta["vollstaendigkeit"]["zeitpunkte"]  # Kreise je Stichtag gegen die Referenzliste (400)

## 4. Cache

Ein zweiter Aufruf innerhalb von 6 Stunden stellt keine Anfrage. `max_age=0` erzwingt die Prüfung des Stands, `force=True` einen neuen Download.

In [ ]:
r2 = fetch(URL, startyear=2024, data_dir=DATEN)
r2.meta["status"]

In [ ]:
assert r2.meta["status"] == "cache"
assert r2.long_df.equals(r.long_df)

## 5. Mehrere Quellen über eine Konfiguration

Ohne eigene Konfiguration nutzt `run()` die mitgelieferte Beispielkonfiguration. Hier eine kleine eigene Konfiguration:

In [ ]:
from pathlib import Path

konfig = Path(DATEN) / "quellen.yaml"
konfig.parent.mkdir(exist_ok=True)
konfig.write_text(f"""
sources:
  - id: bevoelkerung_kreise
    url: {URL}
    auswahl: {{startyear: \"2024\"}}
    rolle: kontext
    aktiv: true
""")

ergebnis = run(config=konfig, data_dir=DATEN)
ergebnis.to_frame()

In [ ]:
assert ergebnis.exit_code == 0
ergebnis.manifest_path

## Weiter

- Die Tabellen liegen als Parquet unter `nb-daten/data/staging/` und lassen sich z. B. mit DuckDB direkt abfragen.
- Optionen und Quellen: README des Repos.